# In-silico patch-clamp recording

This notebook runs a real autoregressive decode through `GPTSynaptic`, records the public `bio-telemetry/1` stream at every generated-token step, and plots per-head presynaptic and postsynaptic state.

In [ ]:
import torch

from bio_inspired_nanochat.gpt_synaptic import GPTSynaptic, GPTSynapticConfig
from bio_inspired_nanochat.patch_clamp import PatchClampElectrode
from bio_inspired_nanochat.synaptic import SynapticConfig

torch.manual_seed(7)
config = GPTSynapticConfig(
    sequence_len=16, vocab_size=64, n_layer=2, n_head=2, n_kv_head=2, n_embd=32,
    synapses=True, use_moe=False,
    syn_cfg=SynapticConfig(enable_presyn=True, enable_hebbian=True),
)
model = GPTSynaptic(config)
electrode = PatchClampElectrode(model)

In [ ]:
prompt = torch.tensor([[1, 5, 9, 13]], dtype=torch.long)
trace = electrode.record_generation(prompt, max_new_tokens=6, temperature=0.0)

print('trace schema:', trace.schema)
print('generated tokens:', trace.generated_token_ids)
print('recorded steps:', trace.time_steps)
print('channel count:', len(trace.channels))
list(sorted(trace.channels))[:12]

In [ ]:
payload = trace.to_dict()
assert payload['schema'] == 'patch-clamp/1'
assert payload['source_schema'] == 'bio-telemetry/1'
assert all(len(channel['values']) == len(trace.time_steps) for channel in payload['channels'].values())

In [ ]:
channels = [
    'L0.attention.H0.calcium',
    'L0.attention.H0.rrp',
    'L0.attention.H0.energy',
    'L0.dense.fc.camkii',
    'L0.dense.fc.pp1',
    'L0.dense.fc.bdnf',
]
electrode.plot_trace(trace, channels)